In [ ]:
df = pd.read_csv("data/train_contrats_anonymized.csv")

In [ ]:
df["freq"] = df["nombre_de_sinistre"] / df["Exposition_au_risque"]

# GLM Model

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_poisson_deviance
from sklearn.linear_model import TweedieRegressor


df = df.copy()
df.columns = df.columns.str.strip().str.lower()

drop = [c for c in ["unnamed: 0", "immat", "num_contrat"] if c in df.columns]
df = df.drop(columns=drop)


features = [
    "annee",
    "classe_age_situ_cont",
    "type_apporteur",
    "creation_entr",
    "mode_gestion",
    "activite",
    "zone",
    "segment",
    "fractionnement",
    "age_du_vehicule",
    "valeurpuissance",
    "franchise",
    "nombre_vehicule",
    "valeur_assuree",
    "formule",
    "categorie_ensemble"
]

X = df[features].copy()


for col in X.columns:
    X[col] = X[col].astype("category")


y = (df["nombre_de_sinistre"] / df["exposition_au_risque"]).astype(float)


expo = df["exposition_au_risque"].astype(float)


X = pd.get_dummies(X, drop_first=True)
X = X.astype(float)



X_train, X_val, y_train, y_val, expo_train, expo_val = train_test_split(
    X, y, expo, test_size=0.2, random_state=42
)





glm = TweedieRegressor(
    power=1,
    alpha=0.0,
    link="log",
    max_iter=1000
)

glm.fit(
    X_train,
    y_train,
    sample_weight=expo_train
)


deviance_train = mean_poisson_deviance(
    y_train,
    glm.predict(X_train),
    sample_weight=expo_train
)

deviance_val = mean_poisson_deviance(
    y_val,
    glm.predict(X_val),
    sample_weight=expo_val
)


print(f"Train deviance: {deviance_train:.4f}")
print(f"Validation deviance: {deviance_val:.4f}")

# 4 neural networks

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_poisson_deviance


tf.random.set_seed(42)
np.random.seed(42)


def build_nn(input_dim, hidden_layers, dropout_rate, l2_reg, learning_rate):
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))

    for u in hidden_layers:
        model.add(
            layers.Dense(
                u,
                activation="tanh",
                kernel_initializer="he_normal",
                kernel_regularizer=regularizers.l2(l2_reg)
            )
        )
        
        model.add(layers.Dropout(dropout_rate))

   
    model.add(layers.Dense(1, activation="exponential"))

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=keras.losses.Poisson()
    )
    return model



nn_configs = {
    "NN1": {
        "hidden_layers": (48,),
        "dropout_rate": 0.0,
        "l2_reg": 0.0,
        "learning_rate": 1e-3
    },

    "NN2": {
        "hidden_layers": (64, 48),
        "dropout_rate": 0.1,
        "l2_reg": 0.0,
        "learning_rate": 1e-3
    },

    "NN3": {
        "hidden_layers": (96, 64),
        "dropout_rate": 0.2,
        "l2_reg": 1e-5,
        "learning_rate": 5e-4
    },

    "NN4": {
        "hidden_layers": (128, 64, 32),
        "dropout_rate": 0.15,
        "l2_reg": 1e-4,
        "learning_rate": 5e-4
    }
}

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

results = []
models = {}

for name, cfg in nn_configs.items():
   
    model = build_nn(
        input_dim=X_train.shape[1],
        hidden_layers=cfg["hidden_layers"],
        dropout_rate=cfg["dropout_rate"],
        l2_reg=cfg["l2_reg"],
        learning_rate=cfg["learning_rate"]
    )

    history = model.fit(
        X_train,
        y_train,
        sample_weight=expo_train,               
        validation_data=(X_val, y_val, expo_val),
        epochs=80,
        batch_size=1000,
        verbose=0,
        callbacks=[early_stop]
    )

 
    deviance_train = mean_poisson_deviance(
        y_train, model.predict(X_train, verbose=0).ravel(), sample_weight=expo_train
    )
    deviance_val = mean_poisson_deviance(
        y_val, model.predict(X_val, verbose=0).ravel(), sample_weight=expo_val
    )

    results.append({
        "model": name,
        "layers": str(cfg["hidden_layers"]),
        "dropout": cfg["dropout_rate"],
        "l2": cfg["l2_reg"],
        "best_val_loss": float(np.min(history.history["val_loss"])),
        "train_deviance": float(deviance_train),
        "val_deviance": float(deviance_val)
    })

    models[name] = model

results_df = pd.DataFrame(results).sort_values("val_deviance")

print(results_df)

#  Compare the predictions of GLM/Neural networks

In [ ]:


best_nn = models["NN1"]  

df_val = df.loc[X_val.index].copy()

df_val["expo"] = expo_val
df_val["claims"] = y_val * expo_val
df_val["freq_obs"] = y_val
df_val["freq_glm"] = glm.predict(X_val)
df_val["freq_nn"] = best_nn.predict(X_val).ravel()

def compare_by_group(df, group_col):
    grouped = df.groupby(group_col, dropna=False).apply(
        lambda g: pd.Series({
            "expo": g["expo"].sum(),
            "claims": g["claims"].sum(),
            "freq_obs": g["claims"].sum() / g["expo"].sum(),
            "freq_glm": (g["freq_glm"] * g["expo"]).sum() / g["expo"].sum(),
            "freq_nn": (g["freq_nn"] * g["expo"]).sum() / g["expo"].sum(),
        })
    ).reset_index()

    return grouped


print(compare_by_group(df_val, "zone"))

print(compare_by_group(df_val, "valeurpuissance"))

print(compare_by_group(df_val, "classe_age_situ_cont"))

print(compare_by_group(df_val, "franchise"))




# Adjust the bias and compare means for some subgroups

In [ ]:
mu_obs = (df_val["claims"].sum() / df_val["expo"].sum())

mu_glm = ((df_val["freq_glm"] * df_val["expo"]).sum()
          / df_val["expo"].sum())

mu_nn = ((df_val["freq_nn"] * df_val["expo"]).sum()
         / df_val["expo"].sum())




df_val["freq_glm_correction"] = (mu_obs / mu_glm)* df_val["freq_glm"]
df_val["freq_nn_correction"] = (mu_obs / mu_nn) * df_val["freq_nn"]

def compare_bias(df, group_col):
    return df.groupby(group_col).apply(
        lambda g: pd.Series({
            "freq_obs": g["claims"].sum() / g["expo"].sum(),

            "GLM_before": (g["freq_glm"] * g["expo"]).sum() / g["expo"].sum(),
            "GLM_after": (g["freq_glm_correction"] * g["expo"]).sum() / g["expo"].sum(),

            "NN_before": (g["freq_nn"] * g["expo"]).sum() / g["expo"].sum(),
            "NN_after": (g["freq_nn_correction"] * g["expo"]).sum() / g["expo"].sum(),
        })
    ).reset_index()



print(compare_bias(df_val, "age_du_vehicule"))

# PDP

In [ ]:
from sklearn.inspection import partial_dependence
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.inspection import PartialDependenceDisplay
import numpy as np
import matplotlib.pyplot as plt
from scikeras.wrappers import KerasRegressor

best_nn = models["NN1"]


class WrappedKerasRegressor(RegressorMixin, BaseEstimator):
    def __init__(self, model):
        self.model = model

    def fit(self, X, y=None):
        self.is_fitted_ = True
        return self

    def predict(self, X):
        X_array = np.asarray(X).astype("float32")
        return self.model.predict(X_array, verbose=0).ravel()

    def __sklearn_is_fitted__(self):
        return True



kr = WrappedKerasRegressor(best_nn)
kr.fit(X_train, y_train)


def pdp_dummy(estimator, X, prefix, ref):

    cols = [c for c in X.columns if c.startswith(prefix)]

    categories = [ref] + [c.replace(prefix, "") for c in cols]

    values = []

    for col in [None] + cols:
        X_tmp = X.copy()
        X_tmp[cols] = 0

        if col is not None:
            X_tmp[col] = 1

        values.append(estimator.predict(X_tmp).mean())

    pdp = pd.DataFrame({
        "category": categories,
        "prediction": values
    })

    plt.figure(figsize=(7,4))
    plt.bar(pdp["category"], pdp["prediction"])
    plt.ylabel("Predicted claim frequency")
    plt.xticks(rotation=45)
    plt.grid(axis="y")
    plt.show()

    return pdp

pdp = pdp_dummy(
    kr,
    X_val,
    prefix="valeurpuissance_",
    ref="1"
)

print(pdp)


# ICE

In [ ]:
feat= ["valeur_assuree_38K - 76K"]

features_ice = [f for f in feat if f in X_val.columns]

PartialDependenceDisplay.from_estimator(
    kr,
    X_val,
    features=features_ice,
    method="brute",
    kind="individual",
    random_state=42
    
)

plt.show()

# LIME

In [ ]:




low = np.argmin(best_nn.predict(X_val, verbose=0).ravel())
high = np.argmax(best_nn.predict(X_val, verbose=0).ravel())

from lime.lime_tabular import LimeTabularExplainer

explainer_lime = LimeTabularExplainer(
    training_data=X_train,
    feature_names=X_train.columns.tolist(),
    mode="regression",
    discretize_continuous=False
)

def predict(x):
    return best_nn.predict(x.astype("float32"), verbose=0).ravel()


lime_low = explainer_lime.explain_instance(
    X_val.iloc[low],
    predict,
    num_features=10
)

fig = lime_low.as_pyplot_figure()
plt.title("LIME - Low risk policy")
plt.show()
lime_high = explainer_lime.explain_instance(
    X_val.iloc[high],
    predict,
    num_features=10
)
fig = lime_high.as_pyplot_figure()
plt.title("LIME - High risk policy")
plt.show()



# Shapley

In [ ]:
import shap



d = X_train.sample(200, random_state=42)

def shap_predict(X):
    X = np.asarray(X).astype("float32")
    return best_nn.predict(X, verbose=0).ravel()

explainer_shap = shap.Explainer(
    shap_predict,
    d,
    feature_names=X_train.columns.tolist()
)

shap_low = explainer_shap(X_val.iloc[[low]])
shap_high = explainer_shap(X_val.iloc[[high]])

shap.plots.waterfall(shap_high[0], max_display=10)
shap.plots.waterfall(shap_low[0], max_display=10)


#  auto-encoders

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split

df.columns = df.columns.str.strip().str.lower()

features_ae = [
    "type_apporteur",
    "creation_entr",
    "mode_gestion",
    "activite",
    "zone",
    "segment",
    "fractionnement",
    "age_du_vehicule",
    "valeurpuissance",
    "formule",
    "categorie_ensemble"
]


X_ae = df[features_ae].copy()

X_ae = pd.get_dummies(X_ae, drop_first=True)
X_ae = X_ae.astype(float)

X_ae = X_ae.values.astype("float32")

X_train_ae, X_val_ae = train_test_split(
    X_ae,
    test_size=0.2,
    random_state=42
)

input_dim = X_train_ae.shape[1]

def reconstruction_mse(model, X_train, X_val):
    recon_train = model.predict(X_train, verbose=0)
    recon_val = model.predict(X_val, verbose=0)

    mse_train = np.mean((X_train - recon_train) ** 2)
    mse_val = np.mean((X_val - recon_val) ** 2)

    return mse_train, mse_val

ae1 = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(32, activation="tanh"),
    layers.Dense(16, activation="tanh"),
    layers.Dense(8, activation="tanh"),
    layers.Dense(16, activation="tanh"),
    layers.Dense(32, activation="tanh"),
    layers.Dense(input_dim, activation="linear")
])

ae1.compile(
    optimizer=keras.optimizers.RMSprop(),
    loss=keras.losses.MeanSquaredError()
)

ae1.fit(
    X_train_ae, X_train_ae,
    validation_data=(X_val_ae, X_val_ae),
    epochs=30,
    batch_size=300,
    verbose=0
    
)

ae1_train_mse, ae1_val_mse = reconstruction_mse(ae1, X_train_ae, X_val_ae)

print("AE1 train MSE:", ae1_train_mse)
print("AE1 val MSE:", ae1_val_mse)

ae2 = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(64, activation="tanh"),
    layers.Dense(40, activation="tanh"),
    layers.Dense(16, activation="tanh"),
    layers.Dense(40, activation="tanh"),
    layers.Dense(64, activation="tanh"),
    layers.Dense(input_dim, activation="linear")
])

ae2.compile(
    optimizer=keras.optimizers.RMSprop(),
    loss=keras.losses.MeanSquaredError()
)

ae2.fit(
    X_train_ae, X_train_ae,
    validation_data=(X_val_ae, X_val_ae),
    epochs=30,
    batch_size=300,
    verbose=0
)

ae2_train_mse, ae2_val_mse = reconstruction_mse(ae2, X_train_ae, X_val_ae)

print("AE2 train MSE:", ae2_train_mse)
print("AE2 val MSE:", ae2_val_mse)

#  variational autoencoders

In [ ]:

input_dim = X_train_ae.shape[1]
latent_dim = 8


inputs = keras.Input(shape=(input_dim,))

x = layers.Dense(64, activation="tanh")(inputs)

z_mean = layers.Dense(latent_dim)(x)
z_log_var = layers.Dense(latent_dim)(x)

def sampling(args):
    z_mean, z_log_var = args
    s = tf.random.normal(shape=tf.shape(z_mean))
    return z_mean + tf.exp(0.5 * z_log_var) * s

z = layers.Lambda(sampling)([z_mean, z_log_var])

encoder = keras.Model(inputs, [z_mean, z_log_var, z])


latent_inputs = keras.Input(shape=(latent_dim,))
x = layers.Dense(64, activation="tanh")(latent_inputs)
outputs = layers.Dense(input_dim, activation="linear")(x)

decoder = keras.Model(latent_inputs, outputs)

class VAE(keras.Model):
    def __init__(self, encoder, decoder):
        super(VAE, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def call(self, inputs):
        z_mean, z_log_var, z = self.encoder(inputs)
        reconstruction = self.decoder(z)
        return reconstruction

    def compute_loss(self, inputs):
        inputs = tf.cast(inputs, tf.float32)

        z_mean, z_log_var, z = self.encoder(inputs)
        reconstruction = self.decoder(z)

        reconstruction_loss = tf.reduce_mean(
            tf.reduce_sum(tf.square(inputs - reconstruction), axis=1)
        )

        kl_loss = -0.5 * tf.reduce_mean(
            tf.reduce_sum(
                1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var),
                axis=1
            )
        )

        return reconstruction_loss + kl_loss
    


vae = VAE(encoder, decoder)

optimizer = keras.optimizers.Adam()

epochs = 30
batch_size = 300

for epoch in range(epochs):
    for step in range(0, X_train_ae.shape[0], batch_size):
        x_batch = X_train_ae[step:step+batch_size]

        with tf.GradientTape() as tape:
            loss = vae.compute_loss(x_batch)

        grads = tape.gradient(loss, vae.trainable_weights)
        optimizer.apply_gradients(zip(grads, vae.trainable_weights))




recon_train = vae(X_train_ae)
recon_val = vae(X_val_ae)

mse_train = np.mean((X_train_ae - recon_train.numpy())**2)
mse_val = np.mean((X_val_ae - recon_val.numpy())**2)

print("VAE train MSE:", mse_train)
print("VAE val MSE:", mse_val)

#  K-means algorithm

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import pandas as pd

outputs = []

for k in range(2, 21):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_ae)

    inertia = kmeans.inertia_
    silhouette = silhouette_score(X_ae , labels)

    outputs.append((k, inertia, silhouette))


outputs_df = pd.DataFrame(outputs, columns=["k", "inertia", "silhouette"])
print(outputs_df)

best_k = outputs_df.loc[outputs_df["silhouette"].idxmax(), "k"]
print("Best k:", best_k)

kmeans = KMeans(n_clusters=int(best_k), random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_ae)

df["cluster"] = clusters

In [ ]:
from sklearn.cluster import KMeans

best_k=2
kmeans = KMeans(n_clusters=int(best_k), random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_ae)

df["cluster"] = clusters

def describe_cluster(df, cluster_id):
    cluster_data = df[df["cluster"] == cluster_id]

    print(f"\nCluster {cluster_id} (size={len(cluster_data)})")

    for col in features_ae:
            
            top = cluster_data[col].value_counts(normalize=True).head(1)
            print(f"{col}: {top.index[0]} ({top.values[0]:.2f})")

for c in range(best_k):
    describe_cluster(df, c)



# Calculate the average claim frequency ,calculate a deviance

In [ ]:
from sklearn.metrics import mean_poisson_deviance

cluster_frequency = df.groupby("cluster").apply(
    lambda g: g["nombre_de_sinistre"].sum() / g["exposition_au_risque"].sum()
).rename("freq_cluster")

print(cluster_frequency)

df["freq_cluster"] = df["cluster"].map(cluster_frequency)



deviance_cluster = mean_poisson_deviance(
    y_true=df["nombre_de_sinistre"] / df["exposition_au_risque"],
    y_pred=df["freq_cluster"],
    sample_weight=df["exposition_au_risque"]
)

print("Cluster deviance:", deviance_cluster)